# Prompt evolution

Runs the EA from Python, same thing `python -m promptevo` does. Docker Desktop and Ollama have to be running.

In [ ]:
import sys
sys.path.append("../src")

from promptevo.datasets import TerminalBench
from promptevo.evaluation import Evaluator, Signal
from promptevo.evolution import Evolution, EvolutionConfig, RunLog
from promptevo.experiment import SEED_PROMPTS, score_test_set, test_prompts
from promptevo.models import mutator, solver
from promptevo.operators import Feedback, LLMOperators
from promptevo.selection import LexicaseSelector, UniformSelector

## Settings

Train items are what the EA sees, test items are only scored at the end. Leave `data_dir` as `None` for the original Terminal-Bench tasks, or point it at `../data/processed/bug-injected/tasks`.

In [ ]:
train = ["risk-scorer-replay"]
test = ["risk-scorer-replay"]
data_dir = None

solver_model = solver("qwen2.5-coder:7b")
mutator_model = mutator("qwen2.5-coder:7b")

seed = 0  # same seed, same run
config = EvolutionConfig(mu=2, lambda_=2, generations=1, seed=seed)
seeds = list(SEED_PROMPTS)
log = RunLog("../results/evolution-notebook.jsonl")

## Run

Swap any piece here, e.g. `Signal.BINARY`, `LexicaseSelector()`, `Feedback.FAILURES`, or another dataset class.

In [ ]:
evolution = Evolution(
    config,
    Evaluator(TerminalBench(train, data_dir), solver_model, Signal.PARTIAL),
    LLMOperators(mutator_model, Feedback.NONE, seed),
    UniformSelector(),
    log,
)
population = evolution.run(seeds)

best = population[0]
print(f"train score {best.evaluation.score:.3f}")
print(best.prompt)

## Test set

Always partial credit. `evolved` vs `best seed` is what the generations added, `no prompt` is the solver with no system prompt.

In [ ]:
tester = Evaluator(TerminalBench(test, data_dir), solver_model)
prompts = test_prompts(best, evolution.seeds)
scores = score_test_set(tester, prompts)
log.test(prompts, scores)
scores